In [48]:
import glob
import sqlite3
import pandas as pd

files = sorted(glob.glob("../../data/raw/weather/*.csv"))
print(len(files))

5


In [49]:
def read_weather(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        print("cp1252:", path)
        return pd.read_csv(path, encoding="cp1252")

weather = pd.concat([read_weather(f) for f in files], ignore_index=True)

In [17]:
weather.head()

,UTC_DATE,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY
0,2026-06-28T00:00:00,22.9,0.0,54
1,2026-06-28T01:00:00,22.2,0.0,57
2,2026-06-28T02:00:00,21.3,0.0,59
3,2026-06-28T03:00:00,20.3,0.0,64
4,2026-06-28T04:00:00,19.4,0.0,68


In [7]:
weather.columns

Index(['x', 'y', 'STATION_NAME', 'CLIMATE_IDENTIFIER', 'ID', 'LOCAL_DATE',
       'PROVINCE_CODE', 'LOCAL_YEAR', 'LOCAL_MONTH', 'LOCAL_DAY', 'LOCAL_HOUR',
       'UTC_DATE', 'UTC_YEAR', 'UTC_MONTH', 'UTC_DAY', 'TEMP', 'TEMP_FLAG',
       'DEW_POINT_TEMP', 'DEW_POINT_TEMP_FLAG', 'HUMIDEX', 'HUMIDEX_FLAG',
       'PRECIP_AMOUNT', 'PRECIP_AMOUNT_FLAG', 'RELATIVE_HUMIDITY',
       'RELATIVE_HUMIDITY_FLAG', 'STATION_PRESSURE', 'STATION_PRESSURE_FLAG',
       'VISIBILITY', 'VISIBILITY_FLAG', 'WEATHER_ENG_DESC', 'WEATHER_FRE_DESC',
       'WINDCHILL', 'WINDCHILL_FLAG', 'WIND_DIRECTION', 'WIND_DIRECTION_FLAG',
       'WIND_SPEED', 'WIND_SPEED_FLAG', 'STN_ID', 'LONGITUDE_DECIMAL_DEGREES',
       'LATITUDE_DECIMAL_DEGREES', 'FLAG'],
      dtype='str')

In [50]:
weather = weather[["UTC_DATE","TEMP","PRECIP_AMOUNT","RELATIVE_HUMIDITY"]]

In [12]:
weather.columns

Index(['UTC_DATE', 'TEMP', 'PRECIP_AMOUNT', 'RELATIVE_HUMIDITY'], dtype='str')

In [51]:
weather["PRECIP_AMOUNT"].isna().sum()

np.int64(0)

In [16]:
weather["PRECIP_AMOUNT"].describe()

count    42933.000000
mean         0.095039
std          0.686542
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         45.000000
Name: PRECIP_AMOUNT, dtype: float64

In [18]:
weather.isna().sum()

UTC_DATE             0
TEMP                 0
PRECIP_AMOUNT        0
RELATIVE_HUMIDITY    0
dtype: int64

In [21]:
weather["RELATIVE_HUMIDITY"].quantile([0.99,0.999,0.9999])

0.9900    95.0
0.9990    98.0
0.9999    99.0
Name: RELATIVE_HUMIDITY, dtype: float64

In [52]:
weather["UTC_DATE"] = pd.to_datetime(weather["UTC_DATE"],utc=True)

In [23]:
weather.head()

,UTC_DATE,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY
0,2026-06-28 00:00:00+00:00,22.9,0.0,54
1,2026-06-28 01:00:00+00:00,22.2,0.0,57
2,2026-06-28 02:00:00+00:00,21.3,0.0,59
3,2026-06-28 03:00:00+00:00,20.3,0.0,64
4,2026-06-28 04:00:00+00:00,19.4,0.0,68


In [53]:
weather.dtypes

UTC_DATE             datetime64[us, UTC]
TEMP                             float64
PRECIP_AMOUNT                    float64
RELATIVE_HUMIDITY                  int64
dtype: object

In [54]:
weather["local_time"] = weather["UTC_DATE"].dt.tz_convert("America/Toronto")

In [55]:
weather.head()

,UTC_DATE,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY,local_time
0,2026-06-28 00:00:00+00:00,22.9,0.0,54,2026-06-27 20:00:00-04:00
1,2026-06-28 01:00:00+00:00,22.2,0.0,57,2026-06-27 21:00:00-04:00
2,2026-06-28 02:00:00+00:00,21.3,0.0,59,2026-06-27 22:00:00-04:00
3,2026-06-28 03:00:00+00:00,20.3,0.0,64,2026-06-27 23:00:00-04:00
4,2026-06-28 04:00:00+00:00,19.4,0.0,68,2026-06-28 00:00:00-04:00


In [56]:
weather = weather[["local_time","TEMP","PRECIP_AMOUNT","RELATIVE_HUMIDITY"]]

In [36]:
weather.tail()

,local_time,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY,hour
42928,2026-06-27 15:00:00-04:00,25.1,0.0,47,2026-27-06-15:00
42929,2026-06-27 16:00:00-04:00,25.1,0.0,46,2026-27-06-16:00
42930,2026-06-27 17:00:00-04:00,25.2,0.0,51,2026-27-06-17:00
42931,2026-06-27 18:00:00-04:00,24.3,0.0,48,2026-27-06-18:00
42932,2026-06-27 19:00:00-04:00,23.6,0.0,51,2026-27-06-19:00


In [57]:
weather["hour"] = weather["local_time"].dt.strftime('%Y-%m-%d-%H:00')

In [58]:
weather.min()

local_time           2021-12-02 09:00:00-05:00
TEMP                                     -20.3
PRECIP_AMOUNT                              0.0
RELATIVE_HUMIDITY                           14
hour                          2021-12-02-09:00
dtype: object

In [59]:
weather.max()

local_time           2026-10-28 00:00:00-04:00
TEMP                                      37.3
PRECIP_AMOUNT                             45.0
RELATIVE_HUMIDITY                           99
hour                          2026-10-28-00:00
dtype: object

In [60]:
weather[weather["hour"].duplicated(keep=False)].sort_values("hour")

,local_time,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY,hour
31052,2022-11-06 01:00:00-04:00,17.4,2.0,86,2022-11-06-01:00
31053,2022-11-06 01:00:00-05:00,16.4,0.3,84,2022-11-06-01:00
19788,2023-11-05 01:00:00-04:00,8.2,0.0,72,2023-11-05-01:00
19789,2023-11-05 01:00:00-05:00,7.7,0.0,72,2023-11-05-01:00
8511,2024-11-03 01:00:00-04:00,5.9,0.0,58,2024-11-03-01:00
8512,2024-11-03 01:00:00-05:00,5.3,0.0,64,2024-11-03-01:00
37231,2025-11-02 01:00:00-04:00,4.9,0.0,76,2025-11-02-01:00
37232,2025-11-02 01:00:00-05:00,5.0,0.0,75,2025-11-02-01:00


In [ ]:
weather.grouby()

In [61]:
len(weather)

42933

In [62]:
weather = (
    weather
    .groupby("hour")
    .agg({
        "TEMP": "mean",
        "PRECIP_AMOUNT": "sum",
        "RELATIVE_HUMIDITY": "mean",
    })
    .reset_index()
)

In [63]:
len(weather)

42929

In [64]:
weather[weather["hour"].duplicated(keep=False)].sort_values("hour")

,hour,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY


In [66]:
weather[weather["hour"] == "2025-11-02-01:00"]

,hour,TEMP,PRECIP_AMOUNT,RELATIVE_HUMIDITY
34295,2025-11-02-01:00,4.95,0.0,75.5


In [72]:
weather = weather[(weather["hour"] >= "2023-01-01-00:00") & (weather["hour"] <= "2026-10-01-01:00")]

In [73]:
len(weather)

32820

In [78]:
import sqlite3

conn = sqlite3.connect("../../data/processed/bikeshare.db")
all_hours = pd.read_sql("SELECT hour FROM all_hours", conn)

missing = all_hours[~all_hours["hour"].isin(weather["hour"])]
print(len(missing))
missing

38


,hour
1682,2023-03-12-02:00
7450,2023-11-07-10:00
7451,2023-11-07-11:00
7452,2023-11-07-12:00
7453,2023-11-07-13:00
7454,2023-11-07-14:00
7765,2023-11-20-13:00
10418,2024-03-10-02:00
13666,2024-07-23-10:00
13667,2024-07-23-11:00


In [83]:
weather = pd.merge(all_hours, weather, on="hour", how="left")

In [87]:
weather["TEMP"] = weather["TEMP"].interpolate(limit=6)

In [89]:
weather["RELATIVE_HUMIDITY"] = weather["RELATIVE_HUMIDITY"].interpolate(limit=6)

Kept hourly temperature, precipitation and humidity. Converted times from UTC to Toronto local time so they line up with trip hours. Combined the duplicate 1 a.m. hours from November daylight-saving changes and dropped the nonexistent 2 a.m. hours in March. Filled 34 missing hours by interpolating temperature and humidity and setting precipitation to 0, with a weather_filled column marking the estimates.

In [94]:
weather["PRECIP_AMOUNT"] = weather["PRECIP_AMOUNT"].fillna(0)

We want to remove those hours that skipped forward, makes no sense for an hour that doesn't exist.